# Part C: Clustering

In this section we apply K-Means clustering to the temporal features engineered from each hotel-month observation. The goal is to discover distinct **demand regimes** — groups of time periods that share similar booking behavior — and translate them into actionable business insights.

**Workflow:**
1. Re-build the monthly observation matrix and features (mirrors Part B)
2. Scale the features
3. Determine optimal K using Elbow Method and Silhouette Score
4. Fit the final K-Means model
5. Profile and interpret each cluster
6. Business recommendations

## C.1 — Imports & Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('Libraries loaded successfully.')

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
df = pd.read_csv('hotel_bookings.csv')

# Encode month as integer (mirrors Part A)
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
df['month_num'] = (pd.Categorical(df['arrival_date_month'],
                                   categories=month_order, ordered=True).codes + 1)

# ── Monthly aggregation per hotel (mirrors Part A & B) ──────────────────────
monthly = (
    df.groupby(['hotel', 'arrival_date_year', 'month_num'])
    .agg(
        total_bookings  = ('is_canceled',               'count'),
        cancellations   = ('is_canceled',               'sum'),
        avg_adr         = ('adr',                       'mean'),
        avg_lead_time   = ('lead_time',                 'mean'),
    )
    .reset_index()
)

monthly['period_idx']   = (monthly['arrival_date_year'] - 2015) * 12 + monthly['month_num']
monthly['cancel_rate']  = monthly['cancellations'] / monthly['total_bookings']
monthly                 = monthly.sort_values(['hotel', 'period_idx']).reset_index(drop=True)

# 3-month rolling average of bookings (smoothed demand signal)
monthly['rolling_bookings'] = (
    monthly.groupby('hotel')['total_bookings']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

print(f'Monthly observations: {len(monthly)}')
print(f'Hotels: {monthly["hotel"].unique()}')
monthly.head(6)

## C.2 — Feature Scaling

K-Means uses **Euclidean distance**, which is sensitive to the scale of each variable. For example, `total_bookings` ranges in the thousands while `cancel_rate` ranges from 0 to 1. Without scaling, high-magnitude features would dominate cluster assignment and mask the contribution of others.

We apply **StandardScaler** (zero mean, unit variance) to all five features:

| Feature | Why included |
|---|---|
| `total_bookings` | Core demand volume — primary clustering signal |
| `avg_adr` | Price level — distinguishes premium vs. budget periods |
| `avg_lead_time` | Booking horizon — reflects customer planning behaviour |
| `cancel_rate` | Risk signal — high-cancel periods have distinct risk profiles |
| `rolling_bookings` | Smoothed trend context — avoids over-reacting to one-off spikes |

In [ ]:
feature_cols = ['total_bookings', 'avg_adr', 'avg_lead_time', 'cancel_rate', 'rolling_bookings']

X = monthly[feature_cols].values

scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Confirm zero mean, unit variance after scaling
scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)
print('After scaling — Mean (should be ~0):')
print(scaled_df.mean().round(6))
print('\nAfter scaling — Std (should be ~1):')
print(scaled_df.std().round(6))

## C.3 — Choosing the Number of Clusters (K)

We use two complementary methods to justify K:

- **Elbow Method**: Plots inertia (within-cluster sum of squares) vs K. The "elbow" point — where additional clusters yield diminishing returns — indicates a good trade-off between fit and simplicity.
- **Silhouette Score**: Measures how similar each observation is to its own cluster vs. neighbouring clusters. Score ranges from -1 (wrong cluster) to +1 (well-separated); higher is better.

Both methods are run for K = 2 through 7.

In [ ]:
K_range   = range(2, 8)
inertias  = []
sil_scores = []

for k in K_range:
    km     = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Choosing Optimal K for K-Means Clustering', fontsize=14, fontweight='bold', y=1.02)

# Elbow
ax1 = axes[0]
ax1.plot(list(K_range), inertias, marker='o', color='#028090', linewidth=2, markersize=8)
ax1.axvline(4, color='#E74C3C', linestyle='--', linewidth=1.5, label='Selected K = 4')
ax1.set_title('Elbow Method — Inertia vs K', fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.legend()
ax1.set_xticks(list(K_range))

# Annotate inertia values
for k, iner in zip(K_range, inertias):
    ax1.annotate(f'{iner:.1f}', xy=(k, iner), xytext=(0, 10),
                 textcoords='offset points', ha='center', fontsize=9, color='#028090')

# Silhouette
ax2 = axes[1]
bar_colors = ['#E74C3C' if k == 4 else '#028090' for k in K_range]
bars = ax2.bar(list(K_range), sil_scores, color=bar_colors, edgecolor='white', linewidth=1.2)
ax2.set_title('Silhouette Score vs K', fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(list(K_range))
ax2.set_ylim(0, 0.55)

# Annotate bars
for bar, score in zip(bars, sil_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
             f'{score:.3f}', ha='center', va='bottom', fontsize=9.5, fontweight='bold')

ax2.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#E74C3C', label='Selected K = 4'),
    plt.Rectangle((0,0),1,1, color='#028090', label='Other K')
], loc='upper right')

plt.tight_layout()
plt.savefig('c3_elbow_silhouette.png', bbox_inches='tight')
plt.show()

print('Inertias   :', dict(zip(K_range, [round(i,1) for i in inertias])))
print('Silhouettes:', dict(zip(K_range, [round(s,3) for s in sil_scores])))

### Justification for K = 4

**Elbow Method:** Inertia drops steeply from K=2 (144.8) to K=3 (99.1) and K=4 (77.1), then flattens noticeably from K=4 onwards (K=5: 59.6, K=6: 45.9, K=7: 40.0). The gradient change between K=3→4 and K=4→5 marks the elbow clearly at **K = 4**.

**Silhouette Score:** The highest score is achieved at **K = 4 (0.416)**. Scores at K=5 (0.395) and K=7 (0.365) are lower, confirming that K=4 produces the best-separated clusters.

Both methods independently converge on **K = 4** as the optimal number of clusters.

## C.4 — Fitting the Final K-Means Model (K = 4)

In [ ]:
km_final = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
monthly['cluster'] = km_final.fit_predict(X_scaled)

final_sil = silhouette_score(X_scaled, monthly['cluster'])
print(f'Final model inertia  : {km_final.inertia_:.2f}')
print(f'Final silhouette score: {final_sil:.4f}')

print('\nCluster sizes:')
print(monthly['cluster'].value_counts().sort_index())

print('\nCluster distribution by hotel:')
print(monthly.groupby(['hotel', 'cluster']).size().unstack(fill_value=0))

## C.5 — Silhouette Plot (per observation)

A per-sample silhouette plot verifies that all clusters are well-separated and no individual observations are significantly misclassified (negative silhouette values would indicate this).

In [ ]:
sil_vals    = silhouette_samples(X_scaled, monthly['cluster'])
cluster_labels = sorted(monthly['cluster'].unique())
cluster_names  = {0: 'Cluster 0\n(Resort Peak)', 1: 'Cluster 1\n(Resort Shoulder)',
                  2: 'Cluster 2\n(City Peak)',    3: 'Cluster 3\n(Off-Season)'}
palette = ['#E74C3C', '#E67E22', '#028090', '#2C3E50']

fig, ax = plt.subplots(figsize=(10, 5))
y_lower = 10

for i, c in enumerate(cluster_labels):
    c_sil   = np.sort(sil_vals[monthly['cluster'] == c])
    size    = len(c_sil)
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, c_sil,
                     facecolor=palette[i], edgecolor='none', alpha=0.85)
    ax.text(-0.05, y_lower + size / 2, str(c), ha='center', va='center',
            fontsize=12, fontweight='bold', color=palette[i])
    y_lower = y_upper + 10

ax.axvline(final_sil, color='black', linestyle='--', linewidth=1.5,
           label=f'Mean silhouette = {final_sil:.3f}')
ax.set_xlabel('Silhouette Coefficient', fontsize=11)
ax.set_ylabel('Cluster')
ax.set_title('Per-Sample Silhouette Plot — K=4', fontweight='bold', fontsize=13)
ax.set_yticks([])
ax.legend(loc='upper right')
ax.set_xlim(-0.15, 0.8)

plt.tight_layout()
plt.savefig('c5_silhouette_plot.png', bbox_inches='tight')
plt.show()

**Interpretation:** All clusters have predominantly positive silhouette coefficients, confirming the observations within each cluster are genuinely more similar to each other than to any other cluster. No cluster shows significant negative values, indicating minimal misclassification.

## C.6 — Cluster Profiles

We now examine the mean feature values for each cluster to understand what makes each group distinct.

In [ ]:
profile = monthly.groupby('cluster')[feature_cols].mean().round(2)

# Assign interpretive names
cluster_names = {
    0: 'Resort Summer Peak',
    1: 'Resort Shoulder Season',
    2: 'City Hotel Peak Demand',
    3: 'Off-Season / Winter Lull'
}
profile['Label'] = profile.index.map(cluster_names)
profile = profile[['Label'] + feature_cols]

print('=== CLUSTER PROFILE (mean feature values) ===')
display(profile)

In [ ]:
# ── Radar / Spider chart of normalised cluster means ─────────────────────────
from matplotlib.patches import FancyArrowPatch

radar_labels   = ['Total\nBookings', 'Avg ADR', 'Lead\nTime', 'Cancel\nRate', 'Rolling\nBookings']
num_vars       = len(radar_labels)
angles         = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles        += angles[:1]   # close the loop

# Normalise profile to [0,1] for radar visibility
prof_vals = profile[feature_cols].copy()
prof_norm = (prof_vals - prof_vals.min()) / (prof_vals.max() - prof_vals.min())

palette   = ['#E74C3C', '#E67E22', '#028090', '#2C3E50']
fig, axes = plt.subplots(2, 2, figsize=(12, 10), subplot_kw=dict(polar=True))
fig.suptitle('Cluster Radar Profiles (Normalised Features)', fontsize=14, fontweight='bold')

for idx, (c, ax) in enumerate(zip([0, 1, 2, 3], axes.flatten())):
    vals = prof_norm.loc[c].values.tolist()
    vals += vals[:1]
    ax.fill(angles, vals, alpha=0.35, color=palette[idx])
    ax.plot(angles, vals, color=palette[idx], linewidth=2)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, fontsize=9)
    ax.set_yticklabels([])
    ax.set_title(f'Cluster {c}\n{cluster_names[c]}',
                 fontsize=11, fontweight='bold', color=palette[idx], pad=15)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('c6_radar_profiles.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Grouped bar chart of cluster means ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Key Feature Differences Across Clusters', fontsize=13, fontweight='bold')

plot_features = [
    ('total_bookings', 'Total Monthly Bookings',  '#028090'),
    ('avg_adr',        'Average Daily Rate (ADR)', '#E74C3C'),
    ('cancel_rate',    'Cancellation Rate',        '#E67E22'),
]
x    = np.arange(4)
cols = [cluster_names[c] for c in [0,1,2,3]]

for ax, (feat, title, color) in zip(axes, plot_features):
    vals = [profile.loc[c, feat] for c in [0,1,2,3]]
    bars = ax.bar(x, vals, color=color, edgecolor='white', linewidth=1.5, alpha=0.88)
    for bar, v in zip(bars, vals):
        fmt = f'{v:.0f}' if v > 10 else f'{v:.2f}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                fmt, ha='center', va='bottom', fontsize=9.5, fontweight='bold')
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels([f'C{c}' for c in [0,1,2,3]], fontsize=10)
    ax.set_xlabel('Cluster')
    ax.set_ylabel(feat.replace('_', ' ').title())

plt.tight_layout()
plt.savefig('c6_feature_bars.png', bbox_inches='tight')
plt.show()

## C.7 — Temporal & Hotel Distribution of Clusters

In [ ]:
month_abbr = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
              7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cluster Distribution — Hotel Type & Seasonal Pattern', fontsize=13, fontweight='bold')

palette_c = {0:'#E74C3C', 1:'#E67E22', 2:'#028090', 3:'#2C3E50'}

# Hotel breakdown
ax1 = axes[0]
hotel_dist = monthly.groupby(['cluster','hotel']).size().unstack(fill_value=0)
hotel_dist.plot(kind='bar', ax=ax1, color=['#1A5276','#2E86C1'], edgecolor='white', width=0.65)
ax1.set_title('Hotel Type per Cluster', fontweight='bold')
ax1.set_xlabel('Cluster')
ax1.set_ylabel('Number of Months')
ax1.set_xticklabels([f'C{c}\n{cluster_names[c][:12]}…' for c in [0,1,2,3]], rotation=0, ha='center', fontsize=8.5)
ax1.legend(title='Hotel Type')

# Seasonal heatmap
ax2 = axes[1]
month_dist = monthly.groupby(['cluster','month_num']).size().unstack(fill_value=0)
month_dist.columns = [month_abbr[m] for m in month_dist.columns]
sns.heatmap(month_dist, ax=ax2, cmap='YlOrRd', annot=True, fmt='d',
            linewidths=0.5, cbar_kws={'label': 'Count (months)'})
ax2.set_title('Monthly Occurrence per Cluster (Heatmap)', fontweight='bold')
ax2.set_xlabel('Month')
ax2.set_ylabel('Cluster')
ax2.set_yticklabels([f'C{c}' for c in month_dist.index], rotation=0)

plt.tight_layout()
plt.savefig('c7_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Cluster membership plotted along the time series ─────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Monthly Booking Demand Coloured by Cluster Assignment', fontsize=13, fontweight='bold')

hotels = ['City Hotel', 'Resort Hotel']

for ax, hotel in zip(axes, hotels):
    sub = monthly[monthly['hotel'] == hotel].sort_values('period_idx').reset_index(drop=True)
    # Background shading per cluster
    for i, row in sub.iterrows():
        ax.axvspan(i - 0.5, i + 0.5, color=palette_c[row['cluster']], alpha=0.25)
    # Demand line
    ax.plot(range(len(sub)), sub['total_bookings'], color='black', linewidth=1.8, label='Total Bookings')
    ax.scatter(range(len(sub)), sub['total_bookings'],
               c=[palette_c[c] for c in sub['cluster']], s=55, zorder=5)
    ax.set_title(hotel, fontweight='bold', fontsize=12)
    ax.set_ylabel('Total Bookings')

    # X ticks = period labels (every 3 months)
    ticks = list(range(0, len(sub), 3))
    labels_x = [f"{month_abbr[sub.loc[t,'month_num']]}\n{int(sub.loc[t,'arrival_date_year'])}" for t in ticks]
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels_x, fontsize=8.5)

# Legend
from matplotlib.lines import Line2D
legend_els = [Line2D([0],[0], marker='o', color='w', markerfacecolor=palette_c[c],
                     markersize=10, label=f'C{c}: {cluster_names[c]}') for c in [0,1,2,3]]
axes[0].legend(handles=legend_els, loc='upper left', fontsize=9, framealpha=0.85)

plt.tight_layout()
plt.savefig('c7_timeseries_clusters.png', bbox_inches='tight')
plt.show()

## C.8 — Cluster Interpretation

Based on the feature profiles and temporal distributions, we assign interpretive labels to each cluster:

---

### 🔴 Cluster 0 — Resort Summer Peak
- **Hotel:** Resort Hotel only (5 months: Jul–Aug)
- **Demand:** Moderate–high (~1,618 bookings/month)
- **Standout:** Highest ADR of all clusters ($177.40) — premium pricing in high season
- **Lead time:** 109 days — guests book well in advance for summer holidays
- **Cancel rate:** 34% — below the dataset average
- **Interpretation:** These are the Resort Hotel's peak summer months. Guests plan far ahead and pay premium prices. Low cancellations suggest committed holidaymakers.

---

### 🟠 Cluster 1 — Resort Shoulder Season
- **Hotel:** Resort Hotel (8 months) + City Hotel (2 months)
- **Demand:** Similar volume to Cluster 0 (~1,732 bookings) but very different ADR ($86.59)
- **Lead time:** Longest of all clusters (135 days) — early planners
- **Cancel rate:** 36%
- **Months:** Spring–early summer (Apr–Jun) and early autumn (Sep)
- **Interpretation:** Shoulder-season resort bookings — guests plan furthest ahead but pay lower rates. Likely price-sensitive leisure travellers taking advantage of off-peak discounts.

---

### 🟢 Cluster 2 — City Hotel Peak Demand
- **Hotel:** City Hotel almost exclusively (20/22 months)
- **Demand:** By far the highest (~3,441 bookings/month) — 2× any other cluster
- **ADR:** Moderate ($108.12)
- **Lead time:** 112 days
- **Cancel rate:** Highest of all clusters (41%)
- **Months:** March–October — a long, broad peak
- **Interpretation:** The City Hotel operates at scale throughout spring–autumn. The very high volume drives revenue despite a high cancellation rate — likely business travellers and conferences with flexible booking policies.

---

### ⚫ Cluster 3 — Off-Season / Winter Lull
- **Hotel:** Resort Hotel (13 months) + City Hotel (4 months)
- **Demand:** Lowest volume (~1,480 bookings/month)
- **ADR:** Lowest of all ($65.85) — heavily discounted off-peak rates
- **Lead time:** Shortest (53 days) — last-minute or spontaneous bookings
- **Cancel rate:** Lowest (25%) — committed guests who specifically chose winter
- **Months:** November–February
- **Interpretation:** Winter low season across both hotel types. Guests book later and pay less. Despite low volume, the low cancel rate suggests higher booking quality (fewer speculative reservations).

## C.9 — Business Recommendations

The four-cluster segmentation reveals actionable demand regimes that should inform distinct operational and commercial strategies:

| Cluster | Strategy |
|---|---|
| **C0 – Resort Summer Peak** | Maximise ADR through dynamic pricing; enforce strict cancellation policies; allocate maximum staffing; upsell ancillary services (spa, dining) to committed summer guests |
| **C1 – Resort Shoulder Season** | Stimulate demand with early-bird packages; offer loyalty rewards to repeat bookers; use the long lead time window to run targeted promotional campaigns 4–5 months ahead |
| **C2 – City Hotel Peak Demand** | Focus on reducing the 41% cancellation rate — introduce non-refundable discounts or deposit requirements; invest in corporate contract accounts that guarantee occupancy; optimise overbooking models to hedge against cancellations |
| **C3 – Off-Season Winter Lull** | Deploy last-minute deal platforms (OTAs, flash sales) matching the 53-day booking horizon; offer bundled packages (e.g., New Year events, ski-adjacent breaks); use the low-cancel reliability to plan maintenance and staff training in guaranteed gaps |

In [ ]:
# ── Final summary table ───────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Cluster':        [0, 1, 2, 3],
    'Label':          list(cluster_names.values()),
    'N Months':       monthly['cluster'].value_counts().sort_index().values,
    'Mean Bookings':  [profile.loc[c,'total_bookings'] for c in [0,1,2,3]],
    'Mean ADR ($)':   [profile.loc[c,'avg_adr']        for c in [0,1,2,3]],
    'Lead Time (d)':  [profile.loc[c,'avg_lead_time']  for c in [0,1,2,3]],
    'Cancel Rate':    [f"{profile.loc[c,'cancel_rate']*100:.1f}%" for c in [0,1,2,3]],
    'Dominant Hotel': ['Resort','Resort/Both','City','Both/Resort'],
    'Dominant Season':['Jul–Aug','Apr–Sep','Mar–Oct','Nov–Feb'],
})
summary = summary.set_index('Cluster')
print('=== FINAL CLUSTER SUMMARY ===')
display(summary)